# Mathematical Statistics (Wackerly, 7th Edition)
## Chapter 3: Discrete Random Variables and Their Probability Distributions

**Textbook:** Mathematical Statistics with Applications, 7th Edition  
**Authors:** Dennis Wackerly, William Mendenhall, Richard Scheaffer

---

### Exercises Covered
- **3.1**  Two well-water impurities — inclusion–exclusion for a discrete distribution
- **3.2**  Two-coin payoff game — mapping a sample space to winnings
- **3.3**  Four components, two defectives — probability the 2nd defective appears on test $y$
- **3.4**  Water-valve system — probability distribution for the number of open paths

For every problem the workflow is the same:
1. **List the support** — what integer values can $Y$ actually take?
2. **Attach a probability to each value** — either by counting equally-likely outcomes or by using inclusion–exclusion / independence.
3. **Check** that $\sum_y p(y) = 1$ and simulate to confirm.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb
from itertools import permutations

rng = np.random.default_rng(20260802)

def plot_pmf(y_vals, p_vals, title, xlabel='y', empirical=None):
    """Bar chart of a discrete distribution, with optional empirical overlay."""
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(y_vals))
    width = 0.38 if empirical is not None else 0.6
    ax.bar(x - (width/2 if empirical is not None else 0), p_vals, width,
           color='#1f4fa8', alpha=0.85, label='exact $p(y)$')
    if empirical is not None:
        ax.bar(x + width/2, empirical, width, color='#a8541f', alpha=0.75, label='simulated')
    for xi, pi in zip(x, p_vals):
        ax.annotate(f'{pi:.3f}', (xi - (width/2 if empirical is not None else 0), pi),
                    ha='center', va='bottom', fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels([str(v) for v in y_vals])
    ax.set_xlabel(xlabel)
    ax.set_ylabel('$P(Y = y)$')
    ax.set_title(title)
    ax.set_ylim(0, max(max(p_vals), max(empirical) if empirical is not None else 0) * 1.18)
    ax.grid(axis='y', alpha=0.3)
    if empirical is not None:
        ax.legend()
    plt.show()

---

## Exercise 3.1 — Two impurities in a well

### The problem

In a county's private wells, 20% have **neither** impurity, 40% have impurity $A$, and 50% have impurity $B$. Some wells have both. Let $Y$ be the number of impurities in a randomly chosen well. Find the probability distribution of $Y$.

### Support of $Y$

There are two impurities, so $Y \in \{0, 1, 2\}$.

### The setup

Let $A$ = the event "well has impurity $A$" and $B$ = "well has impurity $B$".

$$P(A) = 0.40,\qquad P(B) = 0.50,\qquad P(\text{neither}) = P(\overline{A\cup B}) = 0.20.$$

### Step 1 — Get $P(A \cap B)$ by inclusion–exclusion

Having **at least one** impurity is the complement of having neither:

$$P(A \cup B) = 1 - 0.20 = 0.80.$$

Inclusion–exclusion:

$$P(A \cup B) = P(A) + P(B) - P(A \cap B) \;\Longrightarrow\; 0.80 = 0.40 + 0.50 - P(A \cap B).$$

So $P(A \cap B) = 0.10$, i.e. **$P(Y = 2) = 0.10$**.

### Step 2 — Peel off $P(Y = 1)$

Exactly one impurity = (only $A$) + (only $B$):

$$P(Y = 1) = \bigl[P(A) - P(A \cap B)\bigr] + \bigl[P(B) - P(A \cap B)\bigr] = 0.30 + 0.40 = 0.70.$$

Equivalently, $P(Y = 1) = P(A \cup B) - P(A \cap B) = 0.80 - 0.10 = 0.70$.

### Distribution

| $y$ | 0 | 1 | 2 |
|---|---|---|---|
| $p(y)$ | $0.20$ | $0.70$ | $0.10$ |

Sum $= 1.00$. ✓

In [ ]:
# Analytical distribution
PA, PB, P_neither = 0.40, 0.50, 0.20
P_AB   = PA + PB - (1 - P_neither)     # inclusion-exclusion
P_only = (PA - P_AB) + (PB - P_AB)     # exactly one

y_vals = [0, 1, 2]
p_vals = [P_neither, P_only, P_AB]
print('Exact distribution:')
for y, p in zip(y_vals, p_vals):
    print(f'  P(Y = {y}) = {p:.4f}')
print(f'  sum       = {sum(p_vals):.4f}')

# Simulation — draw wells consistent with the four joint cell probabilities
cells = np.array([
    (1 - PA) - (PB - P_AB),   # only-A false, only-B false, both false  = neither
    PA - P_AB,                # only A
    PB - P_AB,                # only B
    P_AB                      # both
])
assert np.isclose(cells.sum(), 1.0), cells
N = 200_000
draws = rng.choice(4, size=N, p=cells)
Y_sim = np.array([0, 1, 1, 2])[draws]
emp = np.array([(Y_sim == y).mean() for y in y_vals])
print('\nSimulated:')
for y, e in zip(y_vals, emp):
    print(f'  P(Y = {y}) ≈ {e:.4f}')

plot_pmf(y_vals, p_vals, 'Exercise 3.1 — impurities in a well', xlabel='y = # impurities', empirical=emp)

---

## Exercise 3.2 — Two-coin payoff game

### The problem

You and a friend each toss a fair coin.

| Outcome | Payoff |
|---|---|
| both tails | $+\$1$ |
| both heads | $+\$2$ |
| one head, one tail | $-\$1$ |

Let $Y$ be your winnings on a single play. Find the pmf.

### Sample space

The four equally-likely outcomes are

$$S = \{HH,\; HT,\; TH,\; TT\},$$

each with probability $\tfrac{1}{4}$.

### Mapping outcomes to $Y$

- $HH \longmapsto Y = 2$   →  probability $1/4$
- $TT \longmapsto Y = 1$   →  probability $1/4$
- $HT$ or $TH \longmapsto Y = -1$   →  probability $2/4 = 1/2$

### Distribution

| $y$ | $-1$ | $1$ | $2$ |
|---|---|---|---|
| $p(y)$ | $0.50$ | $0.25$ | $0.25$ |

Sum $= 1.00$. ✓

### Bonus — expected value and variance

$$E[Y] = (-1)(0.50) + (1)(0.25) + (2)(0.25) = -0.50 + 0.25 + 0.50 = 0.25.$$

So on average you gain **$0.25 per play** — the game is favorable to you.

$$E[Y^2] = (1)(0.50) + (1)(0.25) + (4)(0.25) = 0.50 + 0.25 + 1.00 = 1.75.$$

$$V(Y) = E[Y^2] - (E[Y])^2 = 1.75 - 0.0625 = 1.6875,\qquad \sigma_Y = \sqrt{1.6875} \approx 1.299.$$

In [ ]:
# Enumerate the sample space explicitly
outcomes = [('H', 'H'), ('H', 'T'), ('T', 'H'), ('T', 'T')]
payoff = {('H','H'): 2, ('T','T'): 1, ('H','T'): -1, ('T','H'): -1}

y_vals = [-1, 1, 2]
p_vals = [sum(1 for o in outcomes if payoff[o] == y) / len(outcomes) for y in y_vals]

EY  = sum(y*p for y, p in zip(y_vals, p_vals))
EY2 = sum((y**2)*p for y, p in zip(y_vals, p_vals))
VY  = EY2 - EY**2

print('Exact distribution:')
for y, p in zip(y_vals, p_vals):
    print(f'  P(Y = {y:+d}) = {p:.4f}')
print(f'  sum        = {sum(p_vals):.4f}')
print(f'\nE[Y]  = {EY:.4f}   (average winnings per play)')
print(f'V(Y)  = {VY:.4f}')
print(f'sd(Y) = {np.sqrt(VY):.4f}')

# Simulation
N = 200_000
tosses = rng.integers(0, 2, size=(N, 2))    # 0 = H, 1 = T
Y_sim = np.where((tosses[:,0] == 0) & (tosses[:,1] == 0),  2,
         np.where((tosses[:,0] == 1) & (tosses[:,1] == 1),  1, -1))
emp = np.array([(Y_sim == y).mean() for y in y_vals])
print(f'\nSimulated  E[Y] ≈ {Y_sim.mean():.4f}   V(Y) ≈ {Y_sim.var():.4f}')

plot_pmf(y_vals, p_vals, 'Exercise 3.2 — coin game winnings', xlabel='y = winnings ($)', empirical=emp)

---

## Exercise 3.3 — Testing until the second defective is found

### The problem

Four components contain two defectives. An inspector tests one at a time and stops the moment the second defective is located (but that second defective is still tested). Let $Y$ be the number of tests performed. Find the pmf.

### Support of $Y$

- Two defectives $\Rightarrow Y \ge 2$.
- Only four components exist $\Rightarrow Y \le 4$.
- So $Y \in \{2, 3, 4\}$.

### Where the first-pass count needs a tweak

A tempting shortcut is to write things like $\binom{2}{2}$, $\binom{3}{2}$, $\binom{4}{2}$ over $4!$. But those subsets ignore the **stopping rule**: the test that finishes the process must be the *second* defective, not just *a* defective.

The clean way is to imagine writing the components out in a random test order and asking *where the two defectives fall*. There are $\binom{4}{2} = 6$ equally likely ways to choose the two positions occupied by defectives among the four test slots.

Let $D$ = defective, $G$ = good. The six equally-likely sequences and the value of $Y$ (position of the *second* $D$):

| Sequence | Position of 2nd $D$ | $Y$ |
|---|---|---|
| $DDGG$ | 2 | 2 |
| $DGDG$ | 3 | 3 |
| $GDDG$ | 3 | 3 |
| $DGGD$ | 4 | 4 |
| $GDGD$ | 4 | 4 |
| $GGDD$ | 4 | 4 |

### Counting principle for $Y = k$

For $Y = k$ we need:
- position $k$ is $D$ (the second defective), **and**
- exactly one $D$ among the first $k - 1$ positions.

So the number of favorable sequences is $\binom{k-1}{1} = k - 1$, out of $\binom{4}{2} = 6$ total:

$$P(Y = k) = \frac{k - 1}{6}, \qquad k = 2, 3, 4.$$

### Distribution

| $y$ | $2$ | $3$ | $4$ |
|---|---|---|---|
| $p(y)$ | $1/6 \approx 0.167$ | $2/6 \approx 0.333$ | $3/6 = 0.500$ |

Sum $= 6/6 = 1$. ✓  Notice $Y = 4$ is the **most** likely — you almost always need every test.

In [ ]:
# Enumerate all 4!/2!2! = 6 distinct D/G sequences explicitly.
def all_sequences():
    seen = set()
    for perm in permutations('DDGG'):
        seen.add(perm)
    return sorted(seen)

def second_D_position(seq):
    count = 0
    for i, c in enumerate(seq, start=1):
        if c == 'D':
            count += 1
            if count == 2:
                return i

sequences = all_sequences()
print('All 6 equally-likely arrangements:')
for s in sequences:
    print(f'  {"".join(s)}  ->  Y = {second_D_position(s)}')

y_vals = [2, 3, 4]
p_vals = [(k - 1) / 6 for k in y_vals]
print('\nExact distribution:')
for y, p in zip(y_vals, p_vals):
    print(f'  P(Y = {y}) = {p:.4f}  =  {int(round(p*6))}/6')
print(f'  sum       = {sum(p_vals):.4f}')

# Simulation — random test order over {D, D, G, G}
N = 200_000
deck = np.array(['D', 'D', 'G', 'G'])
Y_sim = np.empty(N, dtype=int)
for i in range(N):
    order = rng.permutation(deck)
    d_positions = np.where(order == 'D')[0]
    Y_sim[i] = d_positions[1] + 1        # 1-indexed position of the second D
emp = np.array([(Y_sim == y).mean() for y in y_vals])
print(f'\nSimulated  E[Y] ≈ {Y_sim.mean():.4f}   (exact = {sum(y*p for y,p in zip(y_vals,p_vals)):.4f})')

plot_pmf(y_vals, p_vals, 'Exercise 3.3 — test number of the 2nd defective',
         xlabel='y = test number', empirical=emp)

---

## Exercise 3.4 — Valves in a water system

### The problem

Water flows from $A$ to $B$ through two parallel branches:

- **Top branch:** a single valve, $V_1$.
- **Bottom branch:** two valves in series, $V_2$ then $V_3$.

Each valve opens independently with probability $0.8$. Let $Y$ = number of *open paths* from $A$ to $B$.

### Support of $Y$

There are only two branches, so $Y \in \{0, 1, 2\}$.

### Reduce the branches to two independent events

Let $T$ = "top branch is open", $L$ = "lower branch is open".

$$P(T) = P(V_1 \text{ open}) = 0.8.$$

$$P(L) = P(V_2 \text{ open}) \cdot P(V_3 \text{ open}) = 0.8 \times 0.8 = 0.64.$$

$T$ depends only on $V_1$, and $L$ depends only on $V_2, V_3$, so $T$ and $L$ are **independent**.

$$P(\bar T) = 0.2, \qquad P(\bar L) = 0.36.$$

### Step 1 — $P(Y = 2)$ (both branches open)

$$P(Y = 2) = P(T)P(L) = 0.8 \times 0.64 = 0.512.$$

### Step 2 — $P(Y = 0)$ (both branches closed)

$$P(Y = 0) = P(\bar T)P(\bar L) = 0.2 \times 0.36 = 0.072.$$

### Step 3 — $P(Y = 1)$ (exactly one branch open)

$$P(Y = 1) = P(T)P(\bar L) + P(\bar T)P(L) = (0.8)(0.36) + (0.2)(0.64) = 0.288 + 0.128 = 0.416.$$

Sanity check: $0.072 + 0.416 + 0.512 = 1.000$. ✓

### Distribution

| $y$ | $0$ | $1$ | $2$ |
|---|---|---|---|
| $p(y)$ | $0.072$ | $0.416$ | $0.512$ |

In [ ]:
p = 0.8
PT = p                        # top branch open
PL = p * p                    # lower branch open (V2 AND V3)
PT_bar = 1 - PT
PL_bar = 1 - PL

P0 = PT_bar * PL_bar
P2 = PT     * PL
P1 = PT * PL_bar + PT_bar * PL

y_vals = [0, 1, 2]
p_vals = [P0, P1, P2]

print(f'P(top open)     = {PT:.3f}')
print(f'P(lower open)   = {PL:.3f}   (0.8 * 0.8)')
print()
print('Exact distribution:')
for y, pv in zip(y_vals, p_vals):
    print(f'  P(Y = {y}) = {pv:.4f}')
print(f'  sum       = {sum(p_vals):.4f}')

# Simulation — draw the three valves independently
N = 500_000
V = rng.random((N, 3)) < p         # True = valve opens
top_open   = V[:, 0]
lower_open = V[:, 1] & V[:, 2]
Y_sim = top_open.astype(int) + lower_open.astype(int)
emp = np.array([(Y_sim == y).mean() for y in y_vals])
print(f'\nSimulated  E[Y] ≈ {Y_sim.mean():.4f}   (exact = {sum(y*pv for y,pv in zip(y_vals,p_vals)):.4f})')

plot_pmf(y_vals, p_vals, 'Exercise 3.4 — number of open A→B paths',
         xlabel='y = open paths', empirical=emp)

### Diagram of the valve system

The picture below matches Wackerly's figure: one valve on the top branch and two valves in series on the bottom branch, both branches running $A \to B$ in parallel.

In [ ]:
from matplotlib.patches import Rectangle, FancyArrowPatch

fig, ax = plt.subplots(figsize=(9, 4.2))

# Nodes A and B
A_xy = (0.4, 1.5)
B_xy = (7.6, 1.5)
for xy, label in [(A_xy, 'A'), (B_xy, 'B')]:
    ax.add_patch(plt.Circle(xy, 0.22, facecolor='#eaeaea', edgecolor='black', linewidth=1.5))
    ax.annotate(label, xy, ha='center', va='center', fontsize=14, fontweight='bold')

def valve(x, y, label):
    ax.add_patch(Rectangle((x - 0.35, y - 0.25), 0.7, 0.5, facecolor='#9ec5fe',
                           edgecolor='black', linewidth=1.5))
    ax.annotate(label, (x, y), ha='center', va='center', fontsize=11, fontweight='bold')

# Top branch: A ---- V1 ---- B  at y = 2.4
y_top = 2.4
ax.plot([A_xy[0]+0.22, 3.6], [A_xy[1], y_top], 'k-', linewidth=1.3)
ax.plot([3.6, 4.4], [y_top, y_top], 'k-', linewidth=1.3)
ax.plot([4.4, B_xy[0]-0.22], [y_top, B_xy[1]], 'k-', linewidth=1.3)
valve(4.0, y_top, r'$V_1$')

# Bottom branch: A ---- V2 ---- V3 ---- B  at y = 0.6
y_bot = 0.6
ax.plot([A_xy[0]+0.22, 2.6], [A_xy[1], y_bot], 'k-', linewidth=1.3)
ax.plot([2.6, 5.4], [y_bot, y_bot], 'k-', linewidth=1.3)
ax.plot([5.4, B_xy[0]-0.22], [y_bot, B_xy[1]], 'k-', linewidth=1.3)
valve(3.0, y_bot, r'$V_2$')
valve(5.0, y_bot, r'$V_3$')

ax.annotate('top branch open $\\iff V_1$ opens\n$P = 0.8$', (4.0, y_top + 0.55),
            ha='center', fontsize=10, color='#1f4fa8')
ax.annotate('lower branch open $\\iff V_2$ AND $V_3$ open\n$P = 0.8 \\times 0.8 = 0.64$',
            (4.0, y_bot - 0.55), ha='center', fontsize=10, color='#a8541f')

ax.set_xlim(-0.2, 8.2)
ax.set_ylim(-0.3, 3.3)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Water valve system: two parallel branches from A to B')
plt.show()

---

## Recap — the pattern across all four exercises

Every one of these problems is the same three-step move:

1. **Support first.** Ask what integer values $Y$ can possibly take. Physical constraints (2 impurities, 2 branches, 4 components...) pin down the finite set.
2. **Attach probabilities.** Use whichever tool the sample space suggests:
   - **3.1** — inclusion–exclusion, because the events overlap and we're only told marginal + complement information.
   - **3.2** — direct enumeration over a 4-point equally-likely sample space.
   - **3.3** — count favorable *ordered* sequences under a stopping rule ("the $k$-th test is the 2nd defective").
   - **3.4** — independence: reduce a compound event to a product, then peel off the three cases.
3. **Verify.** Confirm $\sum p(y) = 1$ and run a simulation to sanity-check the analytical numbers.